# Le Gros Chaton — Trajectory SFT (Phase 2b)

Bakes the teacher's agentic behavior (tool calls, self-awareness, creativity) into the 9B weights.

**What this does:**
1. Clones the le-gros-chaton repo (has the training code)
2. Pulls verified Kimi K3 teacher traces from `mateo0093/le-gros-chaton-traces`
3. Loads Qwen3.5-9B + the 91% SFT adapter (`mateo0093/le-gros-chaton-qwen`)
4. Trains on ASSISTANT tokens only (the model learns to *act*, not copy tool output)
5. Uploads the trajectory-SFT adapter back to HF

**How to run:** Runtime → Change runtime type → **T4 GPU** (free) → Run all.

In [ ]:
# 0. Install deps + clone the repo (run once per session)
!pip install -q torch==2.10.0 transformers==5.14.1 tokenizers==0.22.1 peft bitsandbytes datasets accelerate safetensors tiktoken trl
import os
if not os.path.isdir('le-gros-chaton'):
    !git clone -q https://github.com/mateo0093/le-gros-chaton.git
os.chdir('le-gros-chaton')
print('deps + repo OK')

In [ ]:
# 1. Config — edit these if needed
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")  # set in Colab secrets 🔑
MODEL_NAME = "Qwen/Qwen3.5-9B"
ADAPTER = "mateo0093/le-gros-chaton-qwen"  # 91% Fable5 SFT adapter
TRACES_REPO = "mateo0093/le-gros-chaton-traces"  # verified teacher traces
OUT_REPO = "mateo0093/le-gros-chaton-qwen"  # where the traj-SFT adapter goes
TRAJECTORY_CTX = 16384  # long context for tool-use traces
BATCH = 1  # fits T4 with grad-accum 8 = eff batch 8
EPOCHS = 3  # small dataset -> multiple passes
LR = 2e-4

import os
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
print('config OK')

In [ ]:
# 2. Pull the verified teacher traces
from huggingface_hub import hf_hub_download
import json, os

local = hf_hub_download(repo_id=TRACES_REPO, repo_type="dataset",
                       filename="agent_traces_full.jsonl", token=HF_TOKEN)
traces = [json.loads(l) for l in open(local) if l.strip()]
print(f"Loaded {len(traces)} traces")
print(f"Verified: {sum(1 for t in traces if t.get('verified'))}")
print(f"Sample turns: {traces[0]['turns']}, messages: {len(traces[0]['messages'])}")

In [ ]:
# 3. Load model + 91% SFT adapter in 4-bit (fits T4)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

quant = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4",
)
tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=quant, device_map="auto",
    trust_remote_code=True, torch_dtype=torch.float16,
)
model = PeftModel.from_pretrained(model, ADAPTER)
print("Model + adapter loaded")
print("VRAM:", round(torch.cuda.memory_allocated()/1e9, 2), "GB")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 4. Trajectory SFT — assistant-token-only loss (bakes BEHAVIOR into weights)
import json, os
with open("agent_traces_full.jsonl", "w") as f:
    for t in traces:
        f.write(json.dumps(t) + "\n")

from train_qwen import train_sft

dataset = {"messages": [t["messages"] for t in traces]}
out_dir, rows = train_sft(
    model, tok, dataset,
    out_dir="qwen_traj_sft",
    lr=LR, epochs=EPOCHS, batch_size=BATCH,
    max_length=TRAJECTORY_CTX,
    resume_from_checkpoint=None, start=0, trajectory=True,
)
print(f"trajectory SFT done: {out_dir}")

In [ ]:
# 5. Upload the trajectory-SFT adapter to HF
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
api.upload_folder(
    folder_path="qwen_traj_sft",
    repo_id=OUT_REPO,
    path_in_repo="traj_sft",
    token=HF_TOKEN,
    ignore_patterns=["*.bin", "optimizer.pt"],
)
print(f"✓ Trajectory SFT adapter uploaded to {OUT_REPO}/traj_sft")
print("\nNext: RLVR with --diversity to bake in creativity.")